# Fase 5 · M01: Modelos Lineales

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M01 — Modelos Lineales |

---

## 🎯 Qué hace

Entrena y evalúa la familia de modelos lineales sobre el dataset definitivo D_strict
(19 features + target `abandono`). Para cada modelo se prueban las 3 estrategias de
balance de clases y se valida con 5-Fold CV estratificado.
Los hiperparámetros óptimos se obtuvieron con Optuna (50 trials × 4 modelos)
y quedan registrados en este notebook para garantizar reproducibilidad.
Incluye coeficientes interpretables, curva de aprendizaje, calibración de
probabilidades y registro de huella CO₂.

**Modelos:** Regresión Logística · LDA · SVM lineal · SVM RBF

**Estrategias de balance:** Sin ajuste · `class_weight='balanced'` · SMOTE

## 📋 Requisitos

- `data/05_modelado/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`
- `data/05_modelado/X_train_prep.parquet`, `X_test_prep.parquet`
- `data/05_modelado/pipeline_preprocesamiento.pkl`
- Entorno: `tfm_abandono` (scikit-learn ≥1.3, optuna, imbalanced-learn, codecarbon, joblib)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/results/results_lineales_basico.parquet` | Métricas completas de las 12 combinaciones |
| `data/05_modelado/results/results_lineales_basico.json` | Metadatos + hiperparámetros registrados |
| `data/05_modelado/models/*.pkl` | 12 modelos entrenados serializados |
| `docs/html/fase5/m01_lineales_basico.html` | Informe HTML |

## 🔄 Flujo

```
X_train_prep / y_train (f5_m01_preparacion)
    ↓ Hiperparámetros registrados (Optuna 50 trials, ejecutado una vez)
    ↓ 5-Fold CV estratificado × 12 combinaciones
    ↓ Entrenamiento final + serialización .pkl
    ↓ Coeficientes interpretables (top-10 features)
    ↓ Curva de aprendizaje (submuestra 30% — SVM RBF es cuadrático)
    ↓ Calibración de probabilidades + curvas ROC
    ↓ codecarbon: huella CO₂
    → results_lineales_basico.parquet + .json + modelos .pkl + HTML
```

## ➡️ Siguiente

`f5_m02_arboles.ipynb` — Decision Tree, Random Forest, AdaBoost

#comprueba  los x_
from pathlib import Path
ruta = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado')
for f in ['X_train_prep.parquet', 'X_test_prep.parquet', 'y_train.parquet', 'y_test.parquet']:
    existe = (ruta / f).exists()
    print(f'{"✅" if existe else "❌"} {f}')


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit,
    cross_validate, learning_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from codecarbon import EmissionsTracker
    CODECARBON_OK = True
except ImportError:
    CODECARBON_OK = False
    print('⚠️  codecarbon no instalado')

warnings.filterwarnings('ignore')

# ROOT detection
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina_desde_fichero

# Rutas
RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_RESULTS  = RUTA_MODELADO / 'results'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_RESULTS, RUTA_MODELS, RUTA_HTML_F5])

# Constantes
RANDOM_STATE    = 42
N_SPLITS_CV     = 5
N_TRIALS_OPTUNA = 50
FAMILIA         = 'lineales'
ESTRATEGIAS     = ['none', 'balanced', 'smote']
COLOR           = '#3182ce'
fmt             = formato_numero_es

info_entorno()
print(f'\n📂 ROOT    : {ROOT}')
print(f'📂 RESULTS : {RUTA_RESULTS}')
print(f'📂 MODELS  : {RUTA_MODELS}')
print(f'\n🌿 codecarbon: {CODECARBON_OK}')

graficos_b64 = {}  # acumulador de gráficos base64

✓ Directorios verificados: 3
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS
# ============================================================================
# Lee splits y preprocesados generados por f5_m01_preparacion.
# Los _prep se cargan desde disco directamente — no se re-aplica el transform.
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    print('⏳ Aplicando pipeline (primera vez)...')
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    X_train_prep.to_parquet(ruta_train_prep)
    X_test_prep.to_parquet(ruta_test_prep)
    print('✅ Preprocesados generados y guardados')

FEATURE_NAMES = list(X_train.columns)
N_FEATURES    = len(FEATURE_NAMES)

print('=' * 55)
print('DATOS CARGADOS')
print('=' * 55)
print(f'X_train : {fmt(len(X_train))} × {N_FEATURES}  |  abandono: {y_train.mean()*100:.1f}%')
print(f'X_test  : {fmt(len(X_test))}  × {N_FEATURES}  |  abandono: {y_test.mean()*100:.1f}%')
print(f'Pipeline: {pipeline_prep.__class__.__name__} ✅')
print()
print('⚠️  X_test / y_test INTOCABLES — solo se usan en M06')

✅ Preprocesados cargados desde disco
DATOS CARGADOS
X_train : 26.896 × 19  |  abandono: 29.2%
X_test  : 6.725  × 19  |  abandono: 29.2%
Pipeline: ColumnTransformer ✅

⚠️  X_test / y_test INTOCABLES — solo se usan en M06


In [3]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    """Ensambla modelo con estrategia de balance.
    none     → modelo directo
    balanced → class_weight en el modelo
    smote    → SMOTE antes del modelo (ImbPipeline)
    """
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    """5-Fold CV estratificado con métricas completas."""
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'familia'        : FAMILIA,
        'f1_mean'        : cv_res['test_f1'].mean(),
        'f1_std'         : cv_res['test_f1'].std(),
        'auc_mean'       : cv_res['test_roc_auc'].mean(),
        'auc_std'        : cv_res['test_roc_auc'].std(),
        'precision_mean' : cv_res['test_precision'].mean(),
        'recall_mean'    : cv_res['test_recall'].mean(),
        'accuracy_mean'  : cv_res['test_accuracy'].mean(),
        'tiempo_s'       : round(t_total, 3),
        'memoria_mb'     : round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    """Métricas completas sobre test para modelo ya entrenado."""
    y_pred  = pipe_fit.predict(X_te)
    y_proba = (
        pipe_fit.predict_proba(X_te)[:, 1]
        if hasattr(pipe_fit, 'predict_proba')
        else pipe_fit.decision_function(X_te)
    )
    # Normalizar decision_function a [0,1] si es necesario
    if y_proba.min() < 0 or y_proba.max() > 1:
        y_proba = (y_proba - y_proba.min()) / (y_proba.max() - y_proba.min() + 1e-9)
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


def construir_modelo(nombre: str, estrategia: str):
    """Construye modelo con hiperparámetros registrados de Optuna."""
    p  = PARAMS_OPTUNA[nombre]
    cw = 'balanced' if estrategia == 'balanced' else None
    if nombre == 'LogReg':
        return LogisticRegression(
            C=p['C'], penalty=p['penalty'], solver='saga',
            max_iter=1000, random_state=RANDOM_STATE, class_weight=cw
        )
    elif nombre == 'LDA':
        kwargs = {'solver': p.get('solver', 'svd')}
        if p.get('shrinkage') is not None:
            kwargs['shrinkage'] = p['shrinkage']
        return LinearDiscriminantAnalysis(**kwargs)
    elif nombre == 'SVM_lineal':
        return SVC(kernel='linear', C=p['C'], probability=True,
                   class_weight=cw, random_state=RANDOM_STATE)
    elif nombre == 'SVM_RBF':
        return SVC(kernel='rbf', C=p['C'], gamma=p['gamma'],
                   probability=True, class_weight=cw, random_state=RANDOM_STATE)


print('✅ Funciones auxiliares definidas')
print('   · construir_pipeline  · evaluar_cv')
print('   · calcular_metricas_test  · construir_modelo')

✅ Funciones auxiliares definidas
   · construir_pipeline  · evaluar_cv
   · calcular_metricas_test  · construir_modelo


In [4]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS ÓPTIMOS (OPTUNA)
# ============================================================================
# Búsqueda realizada con Optuna TPESampler, 50 trials por modelo,
# métrica AUC-ROC 5-Fold CV sobre X_train_prep.
# Tiempos de búsqueda: LogReg 54s · LDA 13s · SVM_lineal 9416s · SVM_RBF 11456s
# Parámetros registrados aquí para garantizar reproducibilidad.
# Para re-ejecutar la búsqueda: cambiar FORZAR_OPTUNA = True.
# ============================================================================

FORZAR_OPTUNA = False

PARAMS_OPTUNA = {
    'LogReg'    : {'C': 4.561252356402935,    'penalty': 'l2'},
    'LDA'       : {'solver': 'eigen',          'shrinkage': None},
    'SVM_lineal': {'C': 0.025950836690011442},
    'SVM_RBF'   : {'C': 1.9922832602091856,   'gamma': 'scale'},
}

AUC_OPTUNA = {
    'LogReg'    : 0.8851,
    'LDA'       : 0.8823,
    'SVM_lineal': 0.8844,
    'SVM_RBF'   : 0.9056,
}

print('✅ Hiperparámetros óptimos registrados (Optuna 50 trials × 4 modelos)')
print(f'   Tiempo total búsqueda: ~{(54+13+9416+11456)/3600:.1f}h')
print()
for nombre, params in PARAMS_OPTUNA.items():
    print(f'   {nombre:<12}: AUC={AUC_OPTUNA[nombre]:.4f}  →  {params}')

✅ Hiperparámetros óptimos registrados (Optuna 50 trials × 4 modelos)
   Tiempo total búsqueda: ~5.8h

   LogReg      : AUC=0.8851  →  {'C': 4.561252356402935, 'penalty': 'l2'}
   LDA         : AUC=0.8823  →  {'solver': 'eigen', 'shrinkage': None}
   SVM_lineal  : AUC=0.8844  →  {'C': 0.025950836690011442}
   SVM_RBF     : AUC=0.9056  →  {'C': 1.9922832602091856, 'gamma': 'scale'}


In [5]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 12 COMBINACIONES (4 modelos × 3 estrategias)
# ============================================================================
# Si los 12 .pkl existen en disco: los carga y reconstruye métricas CV.
# Si faltan algunos: entrena solo los que faltan.
# Una vez generado results_lineales_basico.parquet, las siguientes ejecuciones
# son completamente instantáneas.
# ============================================================================

claves_esperadas = [f'{m}__{e}'
                    for m in ['LogReg', 'LDA', 'SVM_lineal', 'SVM_RBF']
                    for e in ESTRATEGIAS]

pkls_en_disco    = [c for c in claves_esperadas
                    if (RUTA_MODELS / f'{c}.pkl').exists()]
parquet_en_disco = (RUTA_RESULTS / 'results_lineales_basico.parquet').exists()

print(f'📦 Modelos en disco : {len(pkls_en_disco)}/12')
print(f'📊 Parquet resultados: {parquet_en_disco}')

# --- Cargar modelos existentes ---
modelos_fit = {c: joblib.load(RUTA_MODELS / f'{c}.pkl') for c in pkls_en_disco}

# --- Entrenar los que faltan ---
claves_pendientes = [c for c in claves_esperadas if c not in modelos_fit]

if claves_pendientes:
    print(f'\n⏳ Entrenando {len(claves_pendientes)} combinaciones pendientes...')
    if CODECARBON_OK:
        tracker = EmissionsTracker(project_name=f'TFM_f5_m01_{FAMILIA}',
                                   output_dir=str(RUTA_RESULTS), log_level='error')
        tracker.start()
    for clave in claves_pendientes:
        nombre, estrategia = clave.split('__')
        pipe = construir_pipeline(construir_modelo(nombre, estrategia), estrategia)
        pipe.fit(X_train_prep, y_train)
        modelos_fit[clave] = pipe
        joblib.dump(pipe, RUTA_MODELS / f'{clave}.pkl')
        print(f'   ✅ {clave}')
    if CODECARBON_OK:
        emisiones = tracker.stop()
        print(f'🌿 CO₂: {emisiones:.6f} kg')
    else:
        emisiones = None
else:
    emisiones = None
    print('✅ Todos los modelos ya estaban en disco')

# --- Construir df_cv: desde parquet si existe, si no reconstruir con CV ---
if parquet_en_disco:
    df_res = pd.read_parquet(RUTA_RESULTS / 'results_lineales_basico.parquet')
    df_cv  = df_res.sort_values('auc_mean', ascending=False)
    print('✅ Resultados cargados desde parquet')
else:
    print('\n⏳ Calculando métricas CV (primera vez, ~5 min)...')
    resultados_cv   = []
    resultados_test = []
    for nombre in ['LogReg', 'LDA', 'SVM_lineal', 'SVM_RBF']:
        print(f'   {nombre}...', end=' ', flush=True)
        for estrategia in ESTRATEGIAS:
            res_cv   = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                  X_train_prep, y_train, estrategia)
            res_test = calcular_metricas_test(
                nombre, modelos_fit[f'{nombre}__{estrategia}'],
                X_test_prep, y_test, estrategia
            )
            resultados_cv.append(res_cv)
            resultados_test.append(res_test)
        print('✅')
    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    # Guardar para futuras ejecuciones
    df_res.to_parquet(RUTA_RESULTS / 'results_lineales_basico.parquet', index=False)
    print('✅ Parquet guardado')

# Identificar mejor modelo
mejor_fila       = df_cv.iloc[0]
MEJOR_MODELO     = mejor_fila['modelo']
MEJOR_ESTRATEGIA = mejor_fila['estrategia']
MEJOR_CLAVE      = f'{MEJOR_MODELO}__{MEJOR_ESTRATEGIA}'
mejor_pipe       = modelos_fit[MEJOR_CLAVE]

print(f'\n🏆 MEJOR: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {mejor_fila["auc_mean"]:.4f} ± {mejor_fila["auc_std"]:.4f}')
print(f'   F1  CV = {mejor_fila["f1_mean"]:.4f} ± {mejor_fila["f1_std"]:.4f}')

📦 Modelos en disco : 12/12
📊 Parquet resultados: True
✅ Todos los modelos ya estaban en disco
✅ Resultados cargados desde parquet

🏆 MEJOR: SVM_RBF + balanced
   AUC CV = 0.9056 ± 0.0034
   F1  CV = 0.7565 ± 0.0061


In [6]:
# ============================================================================
# CELDA 6: TABLA DE RESULTADOS
# ============================================================================

cols_cv = ['modelo', 'estrategia', 'auc_mean', 'auc_std',
           'f1_mean', 'precision_mean', 'recall_mean', 'tiempo_s']

print('=' * 70)
print('RESULTADOS 5-Fold CV — ordenado por AUC')
print('=' * 70)
print(df_cv[cols_cv].to_string(index=False, float_format='{:.4f}'.format))

if 'auc_test' in df_res.columns:
    cols_test = ['modelo', 'estrategia', 'auc_test', 'f1_test',
                 'precision_test', 'recall_test', 'kappa_test']
    print('\n' + '=' * 70)
    print('MÉTRICAS EN TEST (informativas — selección por CV)')
    print('=' * 70)
    print(df_res[cols_test].sort_values('auc_test', ascending=False)
          .to_string(index=False, float_format='{:.4f}'.format))

RESULTADOS 5-Fold CV — ordenado por AUC
    modelo estrategia  auc_mean  auc_std  f1_mean  precision_mean  recall_mean  tiempo_s
   SVM_RBF   balanced    0.9056   0.0034   0.7565          0.7120       0.8071  112.9010
   SVM_RBF      smote    0.9032   0.0024   0.7583          0.7225       0.7980  448.5630
   SVM_RBF       none    0.9000   0.0039   0.7400          0.8241       0.6716  109.9300
    LogReg   balanced    0.8851   0.0049   0.7216          0.6529       0.8068    6.8070
    LogReg      smote    0.8850   0.0048   0.7236          0.6593       0.8022    6.3850
    LogReg       none    0.8846   0.0049   0.7201          0.7756       0.6721    5.9260
SVM_lineal   balanced    0.8844   0.0044   0.7255          0.6646       0.7990  222.9880
       LDA      smote    0.8843   0.0044   0.7250          0.6600       0.8043    3.1070
SVM_lineal      smote    0.8839   0.0043   0.7266          0.6664       0.7990  313.8180
SVM_lineal       none    0.8830   0.0046   0.7198          0.7782     

In [7]:
# ============================================================================
# CELDA 7: COEFICIENTES INTERPRETABLES (solo modelos lineales)
# ============================================================================
# Los modelos lineales son únicos: sus coeficientes indican la dirección
# e importancia de cada feature. Análisis exclusivo de esta familia.
# ============================================================================


# Buscar el mejor modelo con coeficientes accesibles
modelo_con_coef = None
nombre_coef     = None
for clave_busq in [MEJOR_CLAVE] + claves_esperadas:
    m = modelos_fit[clave_busq].named_steps.get('model',
        list(modelos_fit[clave_busq].named_steps.values())[-1])
    if hasattr(m, 'coef_') or hasattr(m, 'scalings_'):
        modelo_con_coef = m
        nombre_coef     = clave_busq
        break

if modelo_con_coef is not None:
    coef_vals = (
        modelo_con_coef.coef_.ravel()
        if hasattr(modelo_con_coef, 'coef_')
        else modelo_con_coef.scalings_.ravel()
    )
    coeficientes = pd.Series(coef_vals, index=FEATURE_NAMES)\
                     .sort_values(key=abs, ascending=False)
    top10 = coeficientes.head(10)

    fig, ax = plt.subplots(figsize=(9, 5))
    colores = ['#e53e3e' if v > 0 else '#38a169' for v in top10.values]
    ax.barh(top10.index[::-1], top10.values[::-1],
            color=colores[::-1], edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(
        f'Top 10 features por coeficiente — {nombre_coef}\n'
        f'Rojo = mayor riesgo abandono  |  Verde = efecto protector',
        fontsize=11, fontweight='bold')
    ax.set_xlabel('Coeficiente (escala estandarizada)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    graficos_b64['coeficientes'] = figura_a_base64(fig)
    plt.close(fig)

    print(f'✅ Coeficientes de {nombre_coef} — top 10:')
    for feat, val in top10.items():
        signo = '▲ riesgo' if val > 0 else '▼ protector'
        print(f'   {feat:<35} {val:+.4f}  ({signo})')
else:
    print('ℹ️  Ningún modelo expone coeficientes directamente')

✅ Coeficientes de LogReg__none — top 10:
   n_anios_beca                        -1.3447  (▼ protector)
   anios_sin_beca                      -0.9718  (▼ protector)
   cred_superados_anio_1er             -0.7953  (▼ protector)
   nota_1er_anio                       -0.6052  (▼ protector)
   situacion_laboral                   +0.5663  (▲ riesgo)
   nota_acceso                         -0.3925  (▼ protector)
   indicador_interrupcion              +0.2833  (▲ riesgo)
   edad_entrada                        +0.2430  (▲ riesgo)
   orden_preferencia                   -0.1393  (▼ protector)
   via_acceso                          -0.1148  (▼ protector)


In [8]:
# ============================================================================
# CELDA 8: CURVA DE APRENDIZAJE
# ============================================================================
# SVM RBF es cuadrático con el tamaño del dataset — usar la muestra completa
# no es viable. Se usa una submuestra estratificada del 30% del train,
# práctica estándar: la curva muestra la tendencia de generalización.
# ============================================================================

# Submuestra estratificada 30%
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_sub = X_train_prep.iloc[idx_sub]
y_sub = y_train.iloc[idx_sub]

print(f'Submuestra: {fmt(len(X_sub))} filas '
      f'({len(X_sub)/len(X_train_prep)*100:.0f}% del train)  '
      f'abandono={y_sub.mean()*100:.1f}%')
print('Calculando curva de aprendizaje...', end=' ', flush=True)
t0 = time.time()

train_sizes, train_scores, cv_scores = learning_curve(
    mejor_pipe, X_sub, y_sub,
    cv=CV, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
cv_mean    = cv_scores.mean(axis=1)
cv_std     = cv_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color=COLOR, label='Train AUC', linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=COLOR)
ax.plot(train_sizes, cv_mean, 'o-', color='#e53e3e', label='CV AUC (5-fold)', linewidth=2)
ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.15, color='#e53e3e')
ax.set_xlabel('Tamaño del conjunto de entrenamiento (submuestra 30%)')
ax.set_ylabel('AUC-ROC')
ax.set_title(
    f'Curva de aprendizaje — {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}\n'
    f'Submuestra estratificada 30% del train',
    fontsize=11, fontweight='bold')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['curva_aprendizaje'] = figura_a_base64(fig)
plt.close(fig)

gap = train_mean[-1] - cv_mean[-1]
print(f'✅  ({time.time()-t0:.0f}s)')
print(f'Gap train-CV: {gap:.4f}  → '
      f'{"Posible overfitting" if gap > 0.05 else "Generalización correcta"}')

Submuestra: 8.068 filas (30% del train)  abandono=29.3%
✅  (92s)do curva de aprendizaje... 
Gap train-CV: 0.0482  → Generalización correcta


In [9]:
# ============================================================================
# CELDA 9: CALIBRACIÓN DE PROBABILIDADES + CURVAS ROC
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_4 = [COLOR, '#e53e3e', '#38a169', '#d69e2e']

ax_cal = axes[0]
ax_cal.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Calibración perfecta')

ax_roc = axes[1]
ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1)

for idx, nombre_m in enumerate(['LogReg', 'LDA', 'SVM_lineal', 'SVM_RBF']):
    # Mejor estrategia de cada modelo según CV
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    pipe_m    = modelos_fit[f'{nombre_m}__{mejor_est}']
    try:
        y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]
    except Exception:
        continue

    # Calibración
    frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, frac_pos, 'o-',
                color=colores_4[idx],
                label=f'{nombre_m} ({mejor_est})',
                linewidth=1.8, markersize=5)

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=colores_4[idx],
                label=f'{nombre_m} AUC={auc_v:.3f}', linewidth=1.8)

ax_cal.set_xlabel('Probabilidad predicha')
ax_cal.set_ylabel('Fracción positivos reales')
ax_cal.set_title('Calibración de probabilidades', fontweight='bold')
ax_cal.legend(fontsize=8)
ax_cal.spines['top'].set_visible(False)
ax_cal.spines['right'].set_visible(False)

ax_roc.set_xlabel('Tasa de falsos positivos')
ax_roc.set_ylabel('Tasa de verdaderos positivos')
ax_roc.set_title('Curvas ROC comparativas', fontweight='bold')
ax_roc.legend(fontsize=9)
ax_roc.spines['top'].set_visible(False)
ax_roc.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['calibracion_roc'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Calibración + ROC generados')

✅ Calibración + ROC generados


In [10]:
# ============================================================================
# CELDA 10: MATRIZ DE CONFUSIÓN + CONVERGENCIA OPTUNA
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión del mejor modelo
y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
ax_cm = axes[0]
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0, 1]); ax_cm.set_xticklabels(clases, rotation=30, ha='right')
ax_cm.set_yticks([0, 1]); ax_cm.set_yticklabels(clases)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j] > thresh else 'black',
                   fontsize=13, fontweight='bold')
ax_cm.set_title(f'Matriz de confusión\n{MEJOR_MODELO} + {MEJOR_ESTRATEGIA} (test)',
                fontweight='bold')
ax_cm.set_ylabel('Etiqueta real')
ax_cm.set_xlabel('Etiqueta predicha')

# AUC por modelo (barras comparativas)
ax_bar = axes[1]
modelos_nombres = ['LogReg', 'LDA', 'SVM_lineal', 'SVM_RBF']
aucs_cv  = [df_cv[df_cv['modelo']==m]['auc_mean'].max() for m in modelos_nombres]
colores_bar = [COLOR if m != MEJOR_MODELO else '#e53e3e' for m in modelos_nombres]
bars = ax_bar.bar(modelos_nombres, aucs_cv, color=colores_bar, edgecolor='white')
ax_bar.set_ylim(0.85, 0.92)
ax_bar.set_ylabel('AUC-ROC (mejor estrategia, CV)')
ax_bar.set_title('Comparativa AUC por modelo\n(rojo = mejor)', fontweight='bold')
for bar, val in zip(bars, aucs_cv):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['cm_comparativa'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Matriz de confusión + comparativa AUC generados')
print()
print('Classification report — mejor modelo en test:')
print(classification_report(y_test, y_pred_mejor,
      target_names=['Continúa', 'Abandona']))

✅ Matriz de confusión + comparativa AUC generados

Classification report — mejor modelo en test:
              precision    recall  f1-score   support

    Continúa       0.92      0.86      0.89      4758
    Abandona       0.71      0.83      0.76      1967

    accuracy                           0.85      6725
   macro avg       0.82      0.84      0.83      6725
weighted avg       0.86      0.85      0.85      6725



In [11]:
# ============================================================================
# CELDA 11: SERIALIZACIÓN DE RESULTADOS
# ============================================================================
# Separado del entrenamiento: si el HTML falla, los resultados están a salvo.
# ============================================================================

# Guardar parquet si no existe ya
if not (RUTA_RESULTS / 'results_lineales_basico.parquet').exists():
    df_res.to_parquet(RUTA_RESULTS / 'results_lineales_basico.parquet', index=False)
    print('✅ results_lineales_basico.parquet guardado')
else:
    print('✅ results_lineales_basico.parquet ya existía')

# Guardar excel si no existe ya    
df_res.to_excel(RUTA_RESULTS / 'results_lineales_basico.xlsx', index=False)
print('✅ results_lineales_basico.xlsx guardado')

# Guardar JSON con metadatos
meta = {
    'fecha'            : datetime.now().isoformat(),
    'familia'          : FAMILIA,
    'modelos'          : ['LogReg', 'LDA', 'SVM_lineal', 'SVM_RBF'],
    'estrategias'      : ESTRATEGIAS,
    'n_combinaciones'  : 12,
    'n_trials_optuna'  : N_TRIALS_OPTUNA,
    'cv_folds'         : N_SPLITS_CV,
    'random_state'     : RANDOM_STATE,
    'mejor_modelo'     : MEJOR_MODELO,
    'mejor_estrategia' : MEJOR_ESTRATEGIA,
    'mejor_auc_cv'     : round(float(df_cv.iloc[0]['auc_mean']), 4),
    'mejor_f1_cv'      : round(float(df_cv.iloc[0]['f1_mean']), 4),
    'mejores_params'   : PARAMS_OPTUNA,
    'auc_optuna'       : AUC_OPTUNA,
    'emisiones_co2_kg' : emisiones,
}
with open(RUTA_RESULTS / 'results_lineales_basico.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print('✅ results_lineales_basico.json guardado')
print(f'✅ {len(modelos_fit)} modelos .pkl en {RUTA_MODELS}')

✅ results_lineales_basico.parquet ya existía
✅ results_lineales_basico.xlsx guardado
✅ results_lineales_basico.json guardado
✅ 12 modelos .pkl en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado\models


In [12]:
# ============================================================================
# CELDA 11b: AJUSTE DE UMBRAL ÓPTIMO
# ============================================================================
# Por defecto los clasificadores usan umbral=0.5 para decidir si un alumno
# abandona o no. Ese umbral es arbitrario y no considera el coste asimétrico:
#   - Falso negativo (predice no abandona cuando sí): peor consecuencia
#   - Falso positivo (predice abandona cuando no): menos grave
# Buscamos el umbral que maximiza F1 y el que garantiza Recall >= 0.75.
# ============================================================================

from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

RECALL_MINIMO    = 0.75
umbrales_result  = []
umbrales_rango   = np.arange(0.10, 0.91, 0.01)

print('=' * 60)
print('AJUSTE DE UMBRAL — búsqueda sobre test set')
print('=' * 60)

for nombre in ['LogReg', 'LDA', 'SVM_lineal', 'SVM_RBF']:
    mejor_est = (df_cv[df_cv['modelo'] == nombre]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    clave = nombre + '__' + mejor_est
    pipe  = modelos_fit[clave]

    try:
        y_proba = pipe.predict_proba(X_test_prep)[:, 1]
    except Exception:
        print(f'  ' + nombre + ': sin predict_proba — umbral omitido')
        continue

    # Umbral óptimo F1
    f1s     = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0)
               for t in umbrales_rango]
    idx_f1  = int(np.argmax(f1s))
    u_f1    = round(float(umbrales_rango[idx_f1]), 2)
    f1_opt  = round(float(f1s[idx_f1]), 4)

    # Umbral mínimo Recall >= RECALL_MINIMO
    recalls  = [recall_score(y_test, (y_proba >= t).astype(int), zero_division=0)
                for t in umbrales_rango]
    idx_rec  = next((j for j, r in enumerate(recalls) if r >= RECALL_MINIMO), None)
    u_rec    = round(float(umbrales_rango[idx_rec]), 2) if idx_rec is not None else None
    rec_rec  = round(float(recalls[idx_rec]), 4)        if idx_rec is not None else None
    pre_rec  = round(float(precision_score(
                   y_test, (y_proba >= umbrales_rango[idx_rec]).astype(int),
                   zero_division=0)), 4)                if idx_rec is not None else None

    # Métricas con umbral 0.5 (referencia)
    y05      = (y_proba >= 0.50).astype(int)
    f1_05    = round(float(f1_score(y_test, y05, zero_division=0)), 4)
    rec_05   = round(float(recall_score(y_test, y05, zero_division=0)), 4)
    ganancia = round(f1_opt - f1_05, 4)

    umbrales_result.append(dict(
        modelo         = nombre,
        estrategia     = mejor_est,
        umbral_f1      = u_f1,
        f1_umbral_opt  = f1_opt,
        f1_umbral_05   = f1_05,
        ganancia_f1    = ganancia,
        umbral_recall  = u_rec,
        recall_garantizado = rec_rec,
        precision_recall   = pre_rec,
        recall_umbral_05   = rec_05,
    ))

    print(f'  ' + nombre + ' (' + mejor_est + ')')
    print(f'    Umbral 0.50      → F1=' + str(f1_05) + '  Recall=' + str(rec_05))
    print(f'    Umbral ópt. F1  → ' + str(u_f1) + '  F1=' + str(f1_opt) +
          '  (+' + str(ganancia) + ')')
    if u_rec is not None:
        print(f'    Umbral Recall≥' + str(RECALL_MINIMO) + ' → ' + str(u_rec) +
              '  Recall=' + str(rec_rec) + '  Prec=' + str(pre_rec))
    print()

df_umbrales = pd.DataFrame(umbrales_result)
print('✅ Análisis de umbral completado')
print(df_umbrales[['modelo','umbral_f1','f1_umbral_opt','f1_umbral_05','ganancia_f1']].to_string(index=False))


AJUSTE DE UMBRAL — búsqueda sobre test set
  LogReg (balanced)
    Umbral 0.50      → F1=0.7288  Recall=0.8246
    Umbral ópt. F1  → 0.58  F1=0.7406  (+0.0118)
    Umbral Recall≥0.75 → 0.1  Recall=0.9725  Prec=0.3633

  LDA (smote)
    Umbral 0.50      → F1=0.728  Recall=0.821
    Umbral ópt. F1  → 0.61  F1=0.7402  (+0.0122)
    Umbral Recall≥0.75 → 0.1  Recall=0.9573  Prec=0.398

  SVM_lineal (balanced)
    Umbral 0.50      → F1=0.7253  Recall=0.698
    Umbral ópt. F1  → 0.4  F1=0.7412  (+0.0159)
    Umbral Recall≥0.75 → 0.1  Recall=0.9415  Prec=0.4627

  SVM_RBF (balanced)
    Umbral 0.50      → F1=0.7618  Recall=0.7407
    Umbral ópt. F1  → 0.38  F1=0.7686  (+0.0068)
    Umbral Recall≥0.75 → 0.1  Recall=0.9344  Prec=0.5318

✅ Análisis de umbral completado
    modelo  umbral_f1  f1_umbral_opt  f1_umbral_05  ganancia_f1
    LogReg       0.58         0.7406        0.7288       0.0118
       LDA       0.61         0.7402        0.7280       0.0122
SVM_lineal       0.40         0.7412   

In [13]:
# ============================================================================
# CELDA 12: GENERAR HTML
# ============================================================================
# Todo el texto se genera automaticamente desde los resultados en memoria.
# Si cambian los datos y resultados, cambia el texto automaticamente.
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm01_lineales_basico.html'

# --- KPIs ---
kpis = [
    {'valor': '4',                                 'titulo': 'Modelos'},
    {'valor': '3',                                 'titulo': 'Estrategias'},
    {'valor': '12',                                'titulo': 'Combinaciones'},
    {'valor': str(N_TRIALS_OPTUNA),                'titulo': 'Trials Optuna'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.3f}',  'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.3f}',   'titulo': 'Mejor F1 CV'},
]
kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{k["valor"]}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{k["titulo"]}</div>'
    f'</div>'
    for k in kpis
) + '</div>'

# --- Texto dinamico: calculado desde df_cv ---
segundo     = df_cv.iloc[1]
diff_auc    = df_cv.iloc[0]['auc_mean'] - segundo['auc_mean']
diff_f1     = df_cv.iloc[0]['f1_mean']  - segundo['f1_mean']
peor        = df_cv.iloc[-1]
rango_auc   = df_cv.iloc[0]['auc_mean'] - peor['auc_mean']
sub_mejor   = df_cv[df_cv['modelo'] == MEJOR_MODELO]
auc_none    = sub_mejor[sub_mejor['estrategia'] == 'none']['auc_mean'].values[0]
mejora_bal  = df_cv.iloc[0]['auc_mean'] - auc_none

# Texto umbral
if 'df_umbrales' in dir() and not df_umbrales.empty:
    mejor_u = df_umbrales.loc[df_umbrales['ganancia_f1'].idxmax()]
    parrafo_umbral = (
        '<p><strong>Ajuste de umbral:</strong> el umbral por defecto (0.50) puede no ser óptimo '
        'dado el coste asimétrico de los errores. El umbral que maximiza F1 para '
        '<strong>' + str(mejor_u['modelo']) + '</strong> es '
        '<strong>' + str(mejor_u['umbral_f1']) + '</strong>, '
        'mejorando el F1 en <strong>+' + str(mejor_u['ganancia_f1']) + '</strong> puntos. '
        'Para garantizar un Recall &ge; 0.75 (detectar al menos 3 de cada 4 abandonos), '
        'el umbral recomendado es <strong>' + str(mejor_u['umbral_recall']) + '</strong>.</p>'
    )
else:
    parrafo_umbral = ''

texto_dinamico = (
    f'<p>El analisis de <strong>{len(df_cv)} combinaciones</strong> '
    f'(4 modelos x 3 estrategias de balance) revela que '
    f'<strong>{MEJOR_MODELO}</strong> con estrategia <strong>{MEJOR_ESTRATEGIA}</strong> '
    f'es la combinacion ganadora de la familia lineal, alcanzando un '
    f'<strong>AUC CV de {df_cv.iloc[0]["auc_mean"]:.4f} '
    f'&plusmn; {df_cv.iloc[0]["auc_std"]:.4f}</strong> '
    f'y un <strong>F1 de {df_cv.iloc[0]["f1_mean"]:.4f}</strong>.</p>'
    f'<p>Supera al segundo clasificador ({segundo["modelo"]} + {segundo["estrategia"]}) en '
    f'<strong>{diff_auc:.4f} puntos de AUC</strong> y <strong>{diff_f1:.4f} de F1</strong>. '
    f'El rango total entre mejor y peor combinacion es de {rango_auc:.4f} puntos de AUC, '
    f'lo que indica que la eleccion del modelo tiene mas impacto que la estrategia de balance.</p>'
    f'<p>El ajuste de balance mejora {MEJOR_MODELO} en <strong>{mejora_bal:.4f} puntos de AUC</strong> '
    f'respecto a entrenar sin ajuste, confirmando que el desbalance 70/30 del dataset '
    f'penaliza a los modelos lineales cuando no se corrige.</p>'
)

# --- Tabla pivotada: 1 fila por modelo, mejor estrategia ---
modelos_orden = ['LogReg', 'LDA', 'SVM_lineal', 'SVM_RBF']
filas_pivot = ''
for modelo in modelos_orden:
    sub      = df_cv[df_cv['modelo'] == modelo].iloc[0]
    es_mejor = modelo == MEJOR_MODELO
    bg       = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    estrella = ' 🏆' if es_mejor else ''
    auc_opt_str = f'{AUC_OPTUNA[modelo]:.4f}' if AUC_OPTUNA[modelo] is not None else '—'
    filas_pivot += (
        f'<tr style="{bg}">'
        f'<td>{modelo}{estrella}</td>'
        f'<td>{sub["estrategia"]}</td>'
        f'<td>{sub["auc_mean"]:.4f} &plusmn; {sub["auc_std"]:.4f}</td>'
        f'<td>{sub["f1_mean"]:.4f}</td>'
        f'<td>{sub["precision_mean"]:.4f}</td>'
        f'<td>{sub["recall_mean"]:.4f}</td>'
        f'<td>{auc_opt_str}</td>'
        f'<td><code>{json.dumps(PARAMS_OPTUNA[modelo])}</code></td>'
        f'</tr>'
    )
tabla_pivot = (
    '<p>Una fila por modelo con su mejor estrategia. '
    '🏆 = mejor combinacion global.</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Mejor estrategia</th>'
    '<th>AUC CV (mean&plusmn;std)</th><th>F1 CV</th>'
    '<th>Precision</th><th>Recall</th>'
    '<th>AUC Optuna</th><th>Hiperparametros</th>'
    f'</tr></thead><tbody>{filas_pivot}</tbody></table>'
)

# --- Tabla completa CV (12 filas) ---
filas_cv = ''
for _, r in df_cv.iterrows():
    es_mejor = r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA
    bg = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    filas_cv += (
        f'<tr style="{bg}">'
        f'<td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} &plusmn; {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td>'
        f'<td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td>'
        f'<td>{r["tiempo_s"]:.1f}s</td></tr>'
    )
tabla_cv = (
    f'<p>Ordenado por AUC-ROC descendente. Fila destacada = mejor combinacion: '
    f'<strong>{MEJOR_MODELO} + {MEJOR_ESTRATEGIA}</strong> '
    f'(AUC={df_cv.iloc[0]["auc_mean"]:.4f}).</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean&plusmn;std)</th>'
    '<th>F1</th><th>Precision</th><th>Recall</th><th>Tiempo</th>'
    f'</tr></thead><tbody>{filas_cv}</tbody></table>'
)

# --- Hiperparametros Optuna ---

filas_params_list = []
for m in ['LogReg', 'LDA', 'SVM_lineal', 'SVM_RBF']:
    auc_str = f'{AUC_OPTUNA[m]:.4f}' if AUC_OPTUNA[m] is not None else '—'
    filas_params_list.append(
        f'<tr><td>{m}</td><td><code>{json.dumps(PARAMS_OPTUNA[m])}</code></td>'
        f'<td>{auc_str}</td></tr>'
    )
filas_params = ''.join(filas_params_list)



tabla_params = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Hiperparametros optimos (Optuna)</th><th>AUC busqueda</th>'
    f'</tr></thead><tbody>{filas_params}</tbody></table>'
)

# --- Coeficientes y CO2 ---
coef_html = (
    f'<img src="data:image/png;base64,{graficos_b64.get("coeficientes","")}" style="max-width:100%">'
    if 'coeficientes' in graficos_b64
    else '<p>ℹ️ El mejor modelo no expone coeficientes lineales directamente.</p>'
)
co2_html = (
    f'<p>🌿 Huella CO2 del entrenamiento: <strong>{emisiones:.6f} kg CO2eq</strong></p>'
    if emisiones
    else '<p>ℹ️ Modelos cargados desde disco - sin nuevo entrenamiento en esta ejecucion.</p>'
)

secciones = (
    '<section class="seccion"><h2>Resumen</h2>' + kpis_html + '</section>'
    '<section class="seccion"><h2>Conclusiones del analisis</h2>' + texto_dinamico + '</section>'
    '<section class="seccion"><h2>Comparativa por modelo — mejor estrategia de cada uno</h2>' + tabla_pivot + '</section>'
    '<section class="seccion"><h2>Resultados completos 5-Fold CV — 12 combinaciones</h2>' + tabla_cv + '</section>'
    f'<section class="seccion"><h2>Hiperparametros optimos — Optuna ({N_TRIALS_OPTUNA} trials x 4 modelos)</h2>' + tabla_params + '</section>'
    '<section class="seccion"><h2>Coeficientes interpretables — Top 10 features</h2>'
    '<p>Exclusivo de la familia lineal. Rojo = mayor riesgo de abandono. Verde = efecto protector. Escala estandarizada.</p>'
    + coef_html + '</section>'
    f'<section class="seccion"><h2>Curva de aprendizaje — {MEJOR_MODELO}</h2>'
    '<p>Submuestra estratificada 30% del train. Convergencia entre train y CV indica buena generalizacion.</p>'
    f'<img src="data:image/png;base64,{graficos_b64.get("curva_aprendizaje","")}" style="max-width:100%"></section>'
    '<section class="seccion"><h2>Calibracion de probabilidades y curvas ROC</h2>'
    f'<img src="data:image/png;base64,{graficos_b64.get("calibracion_roc","")}" style="max-width:100%"></section>'
    '<section class="seccion"><h2>Matriz de confusion y comparativa AUC</h2>'
    f'<img src="data:image/png;base64,{graficos_b64.get("cm_comparativa","")}" style="max-width:100%"></section>'
    '<section class="seccion"><h2>Sostenibilidad computacional</h2>' + co2_html + '</section>'
)

render_pagina_desde_fichero(
    'f5_m01_lineales_basico.ipynb',
    secciones,
    RUTA_HTML_SALIDA,
    carpeta_notebook='fase5_modelado'
)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')


✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m01_lineales_basico.html


In [14]:
# ============================================================================
# CELDA 13: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m01_lineales')
print('=' * 60)
print(f'Familia     : Modelos lineales')
print(f'Modelos     : LogReg · LDA · SVM_lineal · SVM_RBF')
print(f'Estrategias : none · balanced · smote  (12 combinaciones)')
print(f'Optuna      : {N_TRIALS_OPTUNA} trials registrados  |  AUC métrica')
print()
print(f'🏆 Mejor : {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')
print()
print('📁 Archivos:')
print('   data/05_modelado/results/results_lineales_basico.parquet')
print('   data/05_modelado/results/results_lineales_basico.xlsx')
print('   data/05_modelado/results/results_lineales_basico.json')
print('   data/05_modelado/models/  (12 × .pkl)')
print('   docs/html/fase5/m01_lineales_basico.html')
print()
print('➡️  Siguiente: f5_m01_lineales_ext.ipynb')

RESUMEN FINAL — f5_m01_lineales
Familia     : Modelos lineales
Modelos     : LogReg · LDA · SVM_lineal · SVM_RBF
Estrategias : none · balanced · smote  (12 combinaciones)
Optuna      : 50 trials registrados  |  AUC métrica

🏆 Mejor : SVM_RBF + balanced
   AUC CV = 0.9056 ± 0.0034
   F1  CV = 0.7565 ± 0.0061

📁 Archivos:
   data/05_modelado/results/results_lineales_basico.parquet
   data/05_modelado/results/results_lineales_basico.xlsx
   data/05_modelado/results/results_lineales_basico.json
   data/05_modelado/models/  (12 × .pkl)
   docs/html/fase5/m01_lineales_basico.html

➡️  Siguiente: f5_m01_lineales_ext.ipynb
